# Gemini の応答をストリーミングで受け取る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_llm_api/gemini_streaming.ipynb)

NLP クックブック 22-03 ／ LLM の API（Gemini と Claude）

> API キーが必要（Colab 左の 🔑 シークレットに登録）

長い応答を、生成されたそばから表示したい。

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-flash-latest"

for chunk in client.models.generate_content_stream(model=MODEL, contents="日本語の形態素解析の歴史を 300 字程度で説明してください。"):
    print(chunk.text or "", end="", flush=True)

In [ ]:
# 最後のチャンクに使用量が入る。全文も自分でつなげて保存しておく
pieces, last = [], None
for chunk in client.models.generate_content_stream(model=MODEL, contents="「吾輩は猫である」の冒頭を現代風に 3 文で言い換えてください。"):
    pieces.append(chunk.text or "")
    last = chunk
print("".join(pieces))
print("合計トークン:", last.usage_metadata.total_token_count)

- 生成に時間がかかる長い出力では、ストリーミングにすると最初の文字がすぐに表示され、待ち時間を短く感じる。チャット画面を作るときは必須。
- 思考 (thinking) をするモデルでは、思考が終わるまで最初のチャンクが届かない。
- 対話 (`client.chats`) でも `chat.send_message_stream()` で同じようにストリーミングできる。